# 📈 Multi-Market Screener — Colab Quickstart

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/herrrickshaw/global-stock-screener/blob/main/colab_quickstart.ipynb)

Run the 20-market screener toolkit in Google Colab. Ships with ~1yr of cached OHLCV for **20 exchanges** (~40k symbols) so you can screen immediately, then refresh for live data.

**Workflow:** install → clone (with the cached data) → bootstrap → screen.

> ⚠️ Educational / research only. **NOT investment advice.** Screener output is a mechanical filter, not a buy/sell signal. Past performance does not guarantee future returns. Consult a SEBI-registered (or your jurisdiction's) advisor.

## 1. Install dependencies

In [ ]:
# Colab already has pandas/numpy/pyarrow/requests/openpyxl; install the rest:
!pip install -q lmdb xlrd yfinance nsepython bseindia nse feedparser vaderSentiment investpy

## 2. Get the code + the committed cache (Git LFS)
The parquet seeds are stored with Git LFS, so install LFS before cloning to pull the ~1yr OHLCV data.

In [ ]:
!git lfs install
!git clone --depth 1 https://github.com/herrrickshaw/global-stock-screener.git
%cd global-stock-screener
!git lfs pull   # ensure the parquet seeds are real files, not pointers

## 3. Bootstrap — load the cached data into the fast store
`BHAV_CACHE` is where the working copy + NoSQL store live (Colab: `/content/cache`). `bootstrap()` copies the committed seeds there and builds the LMDB store — **no downloads needed**.

In [ ]:
import os

os.environ['BHAV_CACHE'] = '/content/cache'
import warnings; warnings.filterwarnings('ignore')

import screener_kit as kit

kit.bootstrap()              # seeds -> /content/cache + builds the store
print('markets ready:', kit.markets())

## 4. Screen — built-in strategies
11 strategies: `piotroski, coffee_can, magic_formula, bluest_blue_chips, debt_reduction, dividend_yield, golden_crossover, loss_to_profit, garp, darvas, cash_conversion_cycle`.

`min_turnover_usd` pre-filters to liquid names (faster). Every result carries a **Liquidity** tier (High/Medium/Low).

In [ ]:
# Darvas breakouts on liquid Indian stocks
kit.screen('darvas', 'IN', top=15, min_turnover_usd=1_000_000)

In [ ]:
# Cash Conversion Cycle — India (from screener.in) and US (from SEC EDGAR)
kit.custom_screen({'ccc': ('<', 0)}, 'IN', rank_by='ccc', ascending=True, top=10, show=['ccc'])

## 5. Custom screen — your own parameters
Metrics available: `ltp, ret_21/63/126/252, vol_ann, sma50/200, above_200dma, rsi14, dist_52w_high, dist_52w_low, avg_vol_20, vol_ratio, atr14_pct, max_drawdown, ccc` + any fundamentals present.
Pass a rule dict `{metric: (op, value)}` or a `lambda m: ...` predicate.

In [ ]:
# US momentum: uptrend, near highs, strong 6-month, not overbought
kit.custom_screen(
    {'above_200dma': ('==', True), 'rsi14': ('<', 70), 'dist_52w_high': ('<', 8), 'ret_126': ('>', 20)},
    market='US', rank_by='ret_126', top=15, min_turnover_usd=5_000_000,
    show=['ltp', 'ret_126', 'rsi14', 'dist_52w_high'])

## 6. Global view — momentum leaderboard, 5-year scoreboard, liquidity

In [ ]:
import market_performance as mp
import run_global_analysis as rga

rga.load_highlights().head(15)        # global momentum (committed snapshot)

In [ ]:
mp.load()                              # 20-market 5-year performance scoreboard

## 7. (Optional) Refresh for live data
The committed cache is a starting point. To pull fresh bars:
- India uses official NSE+BSE bhavcopy (incremental, no Yahoo).
- Other markets use the yahoo→stooq fallback chain.

(Skip if the cached snapshot is fine — refreshing 20 markets takes a while.)

In [ ]:
# kit.update('IN')      # official bhavcopy, only new trading days
# kit.update('US')      # recent bars via yahoo->stooq
print('uncomment above to refresh a market')

## 8. (Optional) Build the Daily Market Brief
Assembles the full HTML mailer (India screener + CCC + global momentum + other markets + 5y scoreboard) and shows it inline — no tokens, no email needed.

In [ ]:
from IPython.display import HTML

from build_mailer import build

subject, text, html = build()
print(subject)
HTML(html)